# TSTR pilot (ESC-50): is there a usable range?

The thesis question is whether calibrated distribution metrics predict how useful synthetic data
is for training (TSTR: train on synthetic, test on real). Answering it needs many synthetic sets
whose usefulness actually differs. Before generating ~20 new sets, this pilot checks, on three sets
that already exist, that the measurement can tell good synthetic data from bad.

**Design** (fixed on Oct 17, before any TSTR result; `claude/tstr-study-design.md` in the project):

* ESC-50 at 16 kHz, 2.03 s crops. Folds 1-3 train, fold 4 chooses the classifier recipe, fold 5
  measures TSTR accuracy. The recipe is chosen once on real data and then frozen for every
  condition; no early stopping.
* One small CNN on 128-band log-mel (the A-128 front end). Every synthetic set enters as audio
  through that front end, real audio the same way.
* Conditions, 3 seeds each:
  * **Real-full**: all 1,200 training clips, a fresh random crop on every draw (reference only).
  * **Real-matched**: 800 fixed real crops, 16 per class, no re-cropping. The comparison point
    for every synthetic set, which also has 800 fixed clips, 16 per class.
  * **A-128** (expected good): Path A at 128 bands, Griffin-Lim audio.
  * **B raw** (expected middling): raw-waveform diffusion.
  * **B latent** (expected bad): diffusion over DAC's continuous latent.
  * **A-128 native mel** (extra, practical): A-128's generated spectrograms used directly, no
    Griffin-Lim. Not part of the gate.

**Gate** (pre-registered; the two refinements in italics are added here, before running):

1. If Real-full accuracy on fold 5 is below 55%, the classifier is the problem: fix it first.
2. If the three gate sets' TSTR accuracies differ by less than the seed spread, there is no usable
   range: stop and rethink before sampling more sets. *Seed spread is defined as the largest
   seed-to-seed standard deviation among the three sets. In addition, the difference between the
   best and worst set must have a 95% grouped-bootstrap interval that excludes 0 (resampling test
   clips by source recording), so "differ" means beyond the test-set noise too, not only the seed
   noise.*
3. Otherwise: generate the remaining sets and run the full study.

Runtime on a T4: about 20-30 minutes. Set `QUICK_TEST = True` first.

## 1. Setup

In [ ]:
!pip install -q diffusers accelerate soundfile
!pip uninstall -y -q torchao

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys, csv, json, time, subprocess
REPO_DIR = "/content/thesis"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/1jvaldez1/audio-diffusion-synth.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
sys.path.insert(0, REPO_DIR)
import common as C

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from pathlib import Path

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

## 2. Config

The recipe grid and the gate thresholds are fixed here, before anything is trained. Do not change
them after seeing results.

In [ ]:
QUICK_TEST = False          # True: a few minutes end to end, numbers meaningless

cfg = C.Config()                                  # ESC-50: 16 kHz, 2.03 s crops
cfg128 = C.Config(n_mels=128)                     # the classifier's front end (A-128's)
S = f"{cfg.out_root}/samples"
OUT_DIR = Path(f"{cfg.out_root}/tstr/pilot" + ("_quick" if QUICK_TEST else ""))
OUT_DIR.mkdir(parents=True, exist_ok=True)

SETS = {                                          # name -> (sample file, what it holds)
    "A128":     (f"{S}/path_a_128mel.pt",           "mel"),
    "B_raw":    (f"{S}/path_b_wave.pt",             "waveform"),
    "B_latent": (f"{S}/path_b_latent_wave_k96.pt",  "waveform"),
}
GATE_SETS = ["A128", "B_raw", "B_latent"]         # A128_mel (native, no Griffin-Lim) is extra, not in the gate
GL_ITERS = 64

# classifier recipe: chosen once on Real-matched, scored on fold 4, then frozen
GRID = [dict(lr=lr, steps=st, specaug=sa) for lr in (1e-3, 3e-3) for st in (2000, 4000) for sa in (False, True)]
BATCH = 64
N_SEEDS = 3
TEST_CROPS = 3                                    # evenly spaced valid crops per test clip; probabilities averaged per clip

# gate (pre-registered)
MIN_REAL_FULL = 0.55
N_BOOT = 2000

if QUICK_TEST:
    GRID = [dict(lr=3e-3, steps=30, specaug=False), dict(lr=3e-3, steps=30, specaug=True)]
    N_SEEDS, N_BOOT, GL_ITERS = 2, 200, 8
C.seed_everything(cfg.seed)
print(f"{len(GRID)} recipe candidates, {N_SEEDS} seeds; results -> {OUT_DIR}")

## 3. Real data

Training folds 1-3 (1,200 clips), fold 4 for the recipe, fold 5 for TSTR. **Real-matched** takes
16 clips per class at random from the training folds and one fixed crop from each: the middle one
of its valid crops (crops that hold the sound, by the same energy rule as diffusion training).
Validation and test clips get `TEST_CROPS` evenly spaced valid crops each; a clip's prediction is the
average over its crops, so accuracy is per clip, as is usual for ESC-50.

In [ ]:
data = C.load_esc50(cfg)
train_split, val_split, test_split = C.split_by_fold(data, cfg)
class_names, n_classes = data["class_names"], len(data["class_names"])

meta_csv = Path(cfg.data_root) / "ESC-50-master" / "meta" / "esc50.csv"
if meta_csv.exists():
    with open(meta_csv) as f:
        src = np.array([r["src_file"] for r in csv.DictReader(f)])
    test_groups = src[data["folds"].numpy() == cfg.heldout_fold]
else:
    print("ESC-50 metadata not found; the bootstrap groups by clip instead of by source recording")
    test_groups = np.arange(len(test_split["labels"]))
del data

def fixed_crops(split, k):
    # k evenly spaced valid crops per clip -> (crops, labels, clip index)
    w, y, idx = C.reference_crops(split, cfg, k=k, return_clip_index=True)
    return w, y, idx

def middle_crop(split, idx):
    # one fixed crop per selected clip: the middle valid crop
    starts, valid = C.valid_crop_starts(split["waveforms"][idx], cfg)
    out = []
    for j, i in enumerate(idx.tolist()):
        v = torch.nonzero(valid[j]).squeeze(1)
        s = int(starts[v[len(v) // 2]])
        out.append(split["waveforms"][i, :, s:s + cfg.clip_samples])
    return torch.stack(out)

r = np.random.default_rng(cfg.seed)
ytr_all = train_split["labels"].numpy()
matched_idx = torch.from_numpy(np.sort(np.concatenate([r.choice(np.flatnonzero(ytr_all == c), 16, replace=False)
                                                       for c in range(n_classes)])))
matched_wave, matched_y = middle_crop(train_split, matched_idx), train_split["labels"][matched_idx]

val_wave, val_y, val_clip = fixed_crops(val_split, TEST_CROPS)
test_wave, test_y, test_clip = fixed_crops(test_split, TEST_CROPS)
print(f"train {len(train_split['labels'])} clips | Real-matched {len(matched_y)} crops "
      f"| val {len(val_split['labels'])} clips ({len(val_y)} crops) | test {len(test_split['labels'])} clips ({len(test_y)} crops)")

## 4. Front end and the synthetic sets

The A-128 front end (128 mel bands, its saved dB window) turns every set into the classifier's
input. A-128's spectrograms go through Griffin-Lim first, like any audio, reusing the evaluation
notebook's cached inversion when it is current. A provenance check stops the notebook if a set
does not hold 800 clips, 16 per class, at this sample rate and clip length.

In [ ]:
fe = C.MelFrontEnd.load(f"{cfg.out_root}/frontend_128mel.json", cfg128, device=device)

def to_mel(w, batch=128):
    return torch.cat([fe(w[i:i + batch].to(device).float()) for i in range(0, len(w), batch)])

def gl_audio(name, file):
    # Griffin-Lim audio for a mel sample file, shared with evaluate_paths.ipynb's cache when current
    cache = Path(f"{cfg.out_root}/eval/wave_A_mel128.pt")
    stamp = os.path.getmtime(file)
    if cache.exists():
        d = torch.load(cache)
        if d.get("stamp") == stamp:
            print(f"{name}: Griffin-Lim audio from the evaluation cache"); return d["value"]
    print(f"{name}: running Griffin-Lim ({GL_ITERS} iterations)")
    return C.spec_to_wave(C.load_samples(file)["samples"], fe, n_iter=GL_ITERS, seed=cfg.seed)

synth = {}
for name, (file, kind) in SETS.items():
    s = C.load_samples(file)
    counts = torch.bincount(s["labels"], minlength=n_classes)
    problems = []
    if s["config"].get("sample_rate") != cfg.sample_rate: problems.append(f"sample rate {s['config'].get('sample_rate')}")
    if not (counts == 16).all(): problems.append(f"class counts {counts.min().item()}-{counts.max().item()}, expected 16 each")
    if s["meta"].get("guidance", 1.0) != 1.0: problems.append(f"guidance {s['meta']['guidance']}")
    assert not problems, f"{name}: " + "; ".join(problems)
    w = gl_audio(name, file) if kind == "mel" else s["samples"].float().clamp(-1, 1)
    assert w.shape[-1] == cfg.clip_samples, f"{name}: clips of {w.shape[-1]} samples, expected {cfg.clip_samples}"
    synth[name] = (to_mel(w), s["labels"].long())
    if kind == "mel":
        assert s["config"]["n_mels"] == 128, f"{name}: {s['config']['n_mels']} bands; the native-mel condition needs 128"
        synth[name + "_mel"] = (s["samples"].float().clamp(-1, 1).to(device), s["labels"].long())
    print(f"{name:<10} {tuple(synth[name][0].shape)}  commit {s['meta'].get('commit', '-')}  train steps {s['meta'].get('train_steps')}")

X_matched = (to_mel(matched_wave), matched_y)
X_val, X_test = to_mel(val_wave), to_mel(test_wave)

## 5. The classifier

The probe network from the ocean work, unchanged, so the same small classifier is used across the
thesis. Every choice and why:

| Choice | Reason |
|---|---|
| Four blocks of two 3x3 convolutions + batch norm + ReLU (32, 64, 128, 256 channels), 2x2 max-pooling, ~1.2M parameters | A standard small spectrogram CNN. Large enough to learn 50 classes from 800 clips, small enough that 24+ training runs fit in half an hour. TSTR measures the data, not the classifier, so a plain, well-understood network is the point. |
| Average over time, keep the 8 frequency rows, dropout 0.3, one linear layer | Insensitive to where a sound starts in the crop; keeps where its energy sits in frequency, which is what separates many ESC-50 classes. |
| One fixed recipe for every condition, chosen on Real-matched and fold 4 | If each set got its own tuning, differences in TSTR accuracy would mix data quality with tuning luck. Real-matched is the right set to tune on: it has the same size and structure as every synthetic set. |
| Recipe grid: learning rate {1e-3, 3e-3} x steps {2,000, 4,000} x SpecAugment {off, on} | The three settings that matter most for 800 training clips: how fast, how long, and whether to regularise by masking. 8 candidates, one seed each. |
| SpecAugment: one frequency mask (up to 16 bands) and one time mask (up to 24 frames) per clip, filled with the floor value (-1) | The standard augmentation for small spectrogram datasets. Masking with -1 means "silence" in this front end; 0 would insert a mid-level band that never occurs in real data. |
| AdamW (weight decay 1e-4), one-cycle learning rate, fixed step count, final weights | No early stopping: fold 4 already chose the recipe, and fold 5 must not influence training. |
| Each draw: a uniformly random clip of the set (sets are class-balanced); Real-full draws a fresh random valid crop | Matches how each condition is defined: fixed sets are seen exactly as they are; Real-full is the "all the real data we have" reference. |

In [ ]:
class ProbeCNN(nn.Module):
    def __init__(self, n_classes, n_bands=128, width=(32, 64, 128, 256), p_drop=0.3):
        super().__init__()
        layers, c_in = [], 1
        for c in width:
            layers += [nn.Conv2d(c_in, c, 3, padding=1, bias=False), nn.BatchNorm2d(c), nn.ReLU(inplace=True),
                       nn.Conv2d(c, c, 3, padding=1, bias=False), nn.BatchNorm2d(c), nn.ReLU(inplace=True),
                       nn.MaxPool2d(2)]
            c_in = c
        self.features = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.Dropout(p_drop), nn.Linear(width[-1] * (n_bands // 2 ** len(width)), n_classes))

    def forward(self, x):
        return self.head(self.features(x).mean(dim=-1).flatten(1))


fmask = torchaudio.transforms.FrequencyMasking(16, iid_masks=True)
tmask = torchaudio.transforms.TimeMasking(24, iid_masks=True)

# Real-full: fresh random valid crop per draw
full_wave = train_split["waveforms"].to(device)
full_y = train_split["labels"].to(device)
starts_all, valid_all = C.valid_crop_starts(train_split["waveforms"], cfg)
valid_all = valid_all.float().to(device); starts_all = starts_all.to(device)

def batch_real_full(g, n):
    i = torch.randint(len(full_y), (n,), generator=g).to(device)
    s = starts_all[torch.multinomial(valid_all[i], 1).squeeze(1)]
    offs = s[:, None] + torch.arange(cfg.clip_samples, device=device)[None]
    w = torch.gather(full_wave[i, 0], 1, offs).unsqueeze(1)
    return fe(w), full_y[i]

def batch_fixed(X, y):
    X, y = X.to(device), y.to(device)
    def get(g, n):
        i = torch.randint(len(y), (n,), generator=g).to(device)
        return X[i], y[i]
    return get

def train_clf(get_batch, lr, steps, specaug, seed):
    torch.manual_seed(seed)
    model = ProbeCNN(n_classes).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps)
    scaler = torch.amp.GradScaler(enabled=device == "cuda")
    g = torch.Generator().manual_seed(seed)
    model.train()
    for _ in range(steps):
        x, y = get_batch(g, BATCH)
        if specaug:
            x = tmask(fmask(x, mask_value=-1.0), mask_value=-1.0)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=device == "cuda"):
            loss = F.cross_entropy(model(x), y)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
    return model.eval()

@torch.no_grad()
def clip_probs(model, X, clip_idx, batch=256):
    # per-clip class probabilities: softmax averaged over each clip's crops
    p = torch.cat([model(X[i:i + batch]).float().softmax(-1).cpu() for i in range(0, len(X), batch)])
    n = int(clip_idx.max()) + 1
    out = torch.zeros(n, n_classes).index_add_(0, clip_idx, p)
    return (out / torch.bincount(clip_idx, minlength=n)[:, None]).numpy()

val_clip_y = val_split["labels"].numpy(); test_clip_y = test_split["labels"].numpy()
print(f"classifier parameters: {sum(p.numel() for p in ProbeCNN(n_classes).parameters()):,}")

## 6. Choose the recipe (Real-matched, fold 4), then freeze it

Each candidate is trained once on Real-matched and scored on fold 4. The best is kept; a tie goes
to the fewer steps, then no augmentation, then the lower learning rate (the simpler recipe).

In [ ]:
t0 = time.time()
rows = []
for k, rc in enumerate(GRID):
    m = train_clf(batch_fixed(*X_matched), rc["lr"], rc["steps"], rc["specaug"], seed=1000)
    acc = (clip_probs(m, X_val, val_clip).argmax(1) == val_clip_y).mean()
    rows.append({**rc, "fold-4 accuracy": acc})
    print(f"{rc}  fold 4: {acc:.3f}")
grid = pd.DataFrame(rows)
best = grid.sort_values(["fold-4 accuracy", "steps", "specaug", "lr"], ascending=[False, True, True, True]).iloc[0]
RECIPE = dict(lr=float(best["lr"]), steps=int(best["steps"]), specaug=bool(best["specaug"]))
print(f"\nfrozen recipe: {RECIPE}   ({(time.time() - t0) / 60:.1f} min)")

## 7. Train every condition with the frozen recipe, score fold 5

In [ ]:
conditions = {"Real-full": batch_real_full, "Real-matched": batch_fixed(*X_matched),
              **{name: batch_fixed(*xy) for name, xy in synth.items()}}
probs = {}
t0 = time.time()
for name, getter in conditions.items():
    probs[name] = [clip_probs(train_clf(getter, seed=s, **RECIPE), X_test, test_clip) for s in range(N_SEEDS)]
    accs = [(p.argmax(1) == test_clip_y).mean() for p in probs[name]]
    print(f"{name:<14} fold-5 accuracy {np.mean(accs):.3f} +/- {np.std(accs):.3f}  seeds {np.round(accs, 3).tolist()}")
print(f"({(time.time() - t0) / 60:.1f} min)")

ref = np.mean([(p.argmax(1) == test_clip_y).mean() for p in probs["Real-matched"]])
summary = pd.DataFrame([{"condition": n,
                         "accuracy": np.mean([(p.argmax(1) == test_clip_y).mean() for p in ps]),
                         "seed sd": np.std([(p.argmax(1) == test_clip_y).mean() for p in ps]),
                         "vs Real-matched": np.mean([(p.argmax(1) == test_clip_y).mean() for p in ps]) / ref}
                        for n, ps in probs.items()]).set_index("condition")
summary.round(3)

## 8. The gate

In [ ]:
acc = summary["accuracy"]; sd = summary["seed sd"]
g1 = acc["Real-full"] >= MIN_REAL_FULL

gate_acc = acc[GATE_SETS]
ranked = gate_acc.sort_values(ascending=False, kind="stable").index
best_set, worst_set = ranked[0], ranked[-1]
spread = sd[GATE_SETS].max()
g2a = (gate_acc.max() - gate_acc.min()) > spread

# per test clip: fraction of seeds that got it right; bootstrap over source recordings
correct = {n: np.mean([(p.argmax(1) == test_clip_y) for p in probs[n]], axis=0) for n in GATE_SETS}
rng = np.random.default_rng(0)
ug, inv = np.unique(test_groups, return_inverse=True)
members = [np.flatnonzero(inv == j) for j in range(len(ug))]
diffs = []
for _ in range(N_BOOT):
    idx = np.concatenate([members[j] for j in rng.integers(len(ug), size=len(ug))])
    diffs.append(correct[best_set][idx].mean() - correct[worst_set][idx].mean())
lo, hi = np.percentile(diffs, [2.5, 97.5])
g2b = lo > 0

print(f"1. Real-full accuracy {acc['Real-full']:.3f} (needs >= {MIN_REAL_FULL}): {'yes' if g1 else 'NO'}")
print(f"2. gate sets: " + ", ".join(f"{n} {gate_acc[n]:.3f}" for n in GATE_SETS))
print(f"   range {gate_acc.max() - gate_acc.min():.3f} vs seed spread {spread:.3f}: {'yes' if g2a else 'NO'}")
print(f"   {best_set} - {worst_set}: 95% grouped-bootstrap interval [{lo:.3f}, {hi:.3f}] excludes 0: {'yes' if g2b else 'NO'}")
if not g1:
    verdict = "FIX THE CLASSIFIER: Real-full is below the threshold, so TSTR accuracies are not trustworthy yet"
elif not (g2a and g2b):
    verdict = "STOP: no usable range between good and bad synthetic sets; rethink before sampling more sets"
else:
    verdict = "PASS: generate the remaining synthetic sets and run the full study"
print(f"\n=> {verdict}")
print(f"\nExpected order A128 > B_raw > B_latent; observed: {' > '.join(gate_acc.sort_values(ascending=False).index)}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
order = ["Real-full", "Real-matched"] + [n for n in summary.index if n not in ("Real-full", "Real-matched")]
x = np.arange(len(order))
ax.bar(x, summary.loc[order, "accuracy"], yerr=summary.loc[order, "seed sd"], capsize=3,
       color=["#555555", "#888888"] + ["#3b6ea8" if n in GATE_SETS else "#9aa5b1" for n in order[2:]])
ax.axhline(1 / n_classes, color="k", lw=0.8, ls=":", label="chance")
ax.set_xticks(x, order, rotation=20); ax.set_ylabel("fold-5 accuracy (per clip)")
ax.set_title("TSTR pilot: trained on each set (800 clips), tested on real fold 5"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(OUT_DIR / "pilot_accuracy.png", dpi=130); plt.show()

## 9. Save

In [ ]:
commit = subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"]).decode().strip()
out = {"commit": commit, "quick_test": QUICK_TEST, "recipe": RECIPE, "grid": grid.to_dict("records"),
       "summary": summary.round(4).reset_index().to_dict("records"),
       "per_seed": {n: [float((p.argmax(1) == test_clip_y).mean()) for p in ps] for n, ps in probs.items()},
       "gate": {"real_full_ok": bool(g1), "range_vs_seed_spread": bool(g2a), "bootstrap_ci": [float(lo), float(hi)],
                "bootstrap_ok": bool(g2b), "best": best_set, "worst": worst_set, "verdict": verdict},
       "sets": {n: f for n, (f, _) in SETS.items()}}
with open(OUT_DIR / f"pilot_{commit}.json", "w") as f:
    json.dump(out, f, indent=2, default=float)
np.savez(OUT_DIR / f"pilot_probs_{commit}.npz", **{f"{n}_seed{s}": p for n, ps in probs.items() for s, p in enumerate(ps)})
print("saved", OUT_DIR / f"pilot_{commit}.json")

## How to read the result

* **The gate is the result of this notebook.** Whatever it prints is the decision; the thresholds
  were fixed before running.
* **Real-matched vs Real-full** shows how much 800 fixed crops lose against all the real data;
  each synthetic set is judged against Real-matched, never against Real-full.
* **The expected order** (A-128 > B raw > B latent) comes from the distribution metrics. If the
  gate passes but the order differs, that is already informative for the thesis question: the
  metrics and the usefulness disagree.
* **A-128 native mel vs A-128** shows what Griffin-Lim costs for training. It does not enter the
  gate.